# Construct pydantic model from text input

In [1]:
from pydantic_ai import Agent
agent = Agent(model="google-gla:gemini-2.5-flash")

result = await agent.run("Give me an IT employee working in sweden, shortly")
result

AgentRunResult(output='**Erik Johansson**, 32, is a **Senior DevOps Engineer** at a fintech startup in **Stockholm**.\n\nHe holds a Master\'s in Computer Science from KTH Royal Institute of Technology. Erik is highly skilled in cloud infrastructure (AWS, Azure), automation, and maintaining robust CI/CD pipelines.\n\nHe values Sweden\'s flat hierarchies, collaborative work culture, and the daily "fika" (coffee break). Outside of work, he enjoys cycling to his office in Södermalm and spending weekends hiking or skiing in the Swedish nature. He\'s fluent in English, but naturally speaks Swedish with colleagues.')

In [2]:
from pydantic import BaseModel, Field


class EmployeeModel(BaseModel):
    name: str
    age: int
    salary: int = Field(gt=30_000, lt=50_000)
    position: str


result = await agent.run(
    "Give me an IT employee working in sweden", output_type=EmployeeModel
)
result

AgentRunResult(output=EmployeeModel(name='Kalle', age=30, salary=49999, position='IT Consultant'))

In [3]:
result.output

EmployeeModel(name='Kalle', age=30, salary=49999, position='IT Consultant')

In [4]:
result.output.name, result.output.age, result.output.salary

('Kalle', 30, 49999)

In [5]:
result.output.model_dump()

{'name': 'Kalle', 'age': 30, 'salary': 49999, 'position': 'IT Consultant'}

In [6]:
result = await agent.run(
    "Give me ten employees in AI and data engineering fields, so the roles can vary, salary must be between 30000 and 50000",
    output_type=list[EmployeeModel],
)
result

AgentRunResult(output=[EmployeeModel(name='Alice', age=30, salary=45000, position='AI Engineer'), EmployeeModel(name='Bob', age=35, salary=48000, position='Data Engineer'), EmployeeModel(name='Charlie', age=28, salary=40000, position='Machine Learning Engineer'), EmployeeModel(name='David', age=32, salary=42000, position='Data Scientist'), EmployeeModel(name='Eve', age=29, salary=38000, position='AI Researcher'), EmployeeModel(name='Frank', age=38, salary=49000, position='Lead Data Engineer'), EmployeeModel(name='Grace', age=27, salary=35000, position='Junior AI Engineer'), EmployeeModel(name='Heidi', age=33, salary=46000, position='BI Developer'), EmployeeModel(name='Ivan', age=31, salary=43000, position='MLOps Engineer'), EmployeeModel(name='Judy', age=34, salary=47000, position='Big Data Engineer')])

In [7]:
result.output

[EmployeeModel(name='Alice', age=30, salary=45000, position='AI Engineer'),
 EmployeeModel(name='Bob', age=35, salary=48000, position='Data Engineer'),
 EmployeeModel(name='Charlie', age=28, salary=40000, position='Machine Learning Engineer'),
 EmployeeModel(name='David', age=32, salary=42000, position='Data Scientist'),
 EmployeeModel(name='Eve', age=29, salary=38000, position='AI Researcher'),
 EmployeeModel(name='Frank', age=38, salary=49000, position='Lead Data Engineer'),
 EmployeeModel(name='Grace', age=27, salary=35000, position='Junior AI Engineer'),
 EmployeeModel(name='Heidi', age=33, salary=46000, position='BI Developer'),
 EmployeeModel(name='Ivan', age=31, salary=43000, position='MLOps Engineer'),
 EmployeeModel(name='Judy', age=34, salary=47000, position='Big Data Engineer')]

## CV model - a more complex and nested model

In [8]:
class ExperienceModel(BaseModel):
    title: str
    company: str
    description: str
    start_year: int
    end_year: int


class EducationModel(BaseModel):
    title: str
    education_area: str
    school: str
    description: str
    start_year: int
    end_year: int


class CvModel(BaseModel):
    name: str
    age: int
    experiences: list[ExperienceModel]
    educations: list[EducationModel]


result = await agent.run(
    "Create a fake person that is applying for a data engineering job",
    output_type=CvModel,
)
result


AgentRunResult(output=CvModel(name='Alice Johnson', age=30, experiences=[ExperienceModel(title='Senior Data Engineer', company='TechInnovate Solutions', description='Led the design and implementation of scalable ETL pipelines using Apache Spark and AWS Glue. Managed data warehouses on Amazon Redshift, optimizing query performance and ensuring data integrity. Collaborated with data scientists to provide robust data infrastructure for machine learning models.', start_year=2021, end_year=2024), ExperienceModel(title='Data Engineer', company='DataStream Analytics', description='Developed and maintained data ingestion systems for various sources using Python and Airflow. Built dimensional data models and designed schema for analytical databases. Participated in migration projects from on-premise to cloud data solutions.', start_year=2018, end_year=2021)], educations=[EducationModel(title='Master of Science in Data Science', education_area='Data Science', school='State University of Technolo

In [9]:
result.output.name

'Alice Johnson'

In [10]:
result.output.age

30

In [18]:
result.output.experiences[1].title, result.output.experiences[1].start_year

('Data Engineer', 2018)

## (optional) Postprocessing - load into duckdb and unnesting

This part is optional, but a way to unnest the data and store it could be to use dlt to load the data into duckdb, followed by joining and unnesting.

Other approach could be to store into nosql such as mongodb.

In [ ]:
import dlt

pipeline = dlt.pipeline(
    pipeline_name="cv_json_duckdb",
    destination=dlt.destinations.duckdb("cv.duckdb"),
    dataset_name="staging",
)

# Modeldump = Gör det till dict, 
info = pipeline.run(
    data=[result.output.model_dump()], loader_file_format="jsonl", table_name="cv_entries"
)

print(info)


Pipeline cv_json_duckdb load step completed in 0.32 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\eriku\Desktop\DataenginerSTI2024-2026\Github\AIgeneering_four_week_course\07_pydanticai\Kokchun\cv.duckdb location to store data
Load package 1764177512.8152063 is LOADED and contains no failed jobs


In [21]:
import duckdb 

with duckdb.connect("cv.duckdb") as conn:
    desc = conn.sql("desc;").df()
    cv_entries = conn.sql("FROM staging.cv_entries").df()
    educations = conn.sql("FROM staging.cv_entries__educations").df()
    experiences = conn.sql("FROM staging.cv_entries__experiences").df()

desc

,database,schema,name,column_names,column_types,temporary
0,cv,staging,_dlt_loads,"[load_id, schema_name, status, inserted_at, sc...","[VARCHAR, VARCHAR, BIGINT, TIMESTAMP WITH TIME...",False
1,cv,staging,_dlt_pipeline_state,"[version, engine_version, pipeline_name, state...","[BIGINT, BIGINT, VARCHAR, VARCHAR, TIMESTAMP W...",False
2,cv,staging,_dlt_version,"[version, engine_version, inserted_at, schema_...","[BIGINT, BIGINT, TIMESTAMP WITH TIME ZONE, VAR...",False
3,cv,staging,cv_entries,"[name, age, _dlt_load_id, _dlt_id]","[VARCHAR, BIGINT, VARCHAR, VARCHAR]",False
4,cv,staging,cv_entries__educations,"[title, education_area, school, description, s...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, BIGINT, B...",False
5,cv,staging,cv_entries__experiences,"[title, company, description, start_year, end_...","[VARCHAR, VARCHAR, VARCHAR, BIGINT, BIGINT, VA...",False


In [22]:
cv_entries

,name,age,_dlt_load_id,_dlt_id
0,Alice Johnson,30,1764177512.8152063,SL08oihOkixoyQ


In [23]:
educations

,title,education_area,school,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,Master of Science in Data Science,Data Science,State University of Technology,"Specialized in big data technologies, machine ...",2017,2018,SL08oihOkixoyQ,0,j5Y5j8zfhI0iVg
1,Bachelor of Science in Computer Science,Computer Science,City College of Engineering,"Focused on algorithms, data structures, databa...",2014,2017,SL08oihOkixoyQ,1,+CFvwxGAxUCjuw


In [24]:
experiences

,title,company,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,Senior Data Engineer,TechInnovate Solutions,Led the design and implementation of scalable ...,2021,2024,SL08oihOkixoyQ,0,RwT8M70QBW07gQ
1,Data Engineer,DataStream Analytics,Developed and maintained data ingestion system...,2018,2021,SL08oihOkixoyQ,1,3ZHM/mbuThrojg


In [25]:
duckdb.sql("""
    SELECT 
        cv.name, 
        cv.age, 
        ex.company,
        ex.description AS experience_description,
        ex.start_year AS experience_start_year,
        ex.end_year AS experience_end_year,
        e.title,
        e.education_area,
        e.school,
        e.start_year AS education_start_year,
        e.end_year AS education_end_year
    FROM cv_entries cv
    LEFT JOIN educations e ON cv._dlt_id = e._dlt_parent_id
    LEFT JOIN experiences ex ON cv._dlt_id = ex._dlt_parent_id
    

""").df()

,name,age,company,experience_description,experience_start_year,experience_end_year,title,education_area,school,education_start_year,education_end_year
0,Alice Johnson,30,DataStream Analytics,Developed and maintained data ingestion system...,2018,2021,Master of Science in Data Science,Data Science,State University of Technology,2017,2018
1,Alice Johnson,30,DataStream Analytics,Developed and maintained data ingestion system...,2018,2021,Bachelor of Science in Computer Science,Computer Science,City College of Engineering,2014,2017
2,Alice Johnson,30,TechInnovate Solutions,Led the design and implementation of scalable ...,2021,2024,Master of Science in Data Science,Data Science,State University of Technology,2017,2018
3,Alice Johnson,30,TechInnovate Solutions,Led the design and implementation of scalable ...,2021,2024,Bachelor of Science in Computer Science,Computer Science,City College of Engineering,2014,2017
